In [13]:
import os
import subprocess

configs_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"
python = "/home/pfuerst/venvs/NNMFit0.5_py3.11/bin/python"
script = "/home/pfuerst/venvs/NNMFit0.5_py3.11/bin/calculate_graph.py"

In [14]:
main_out_dir = f"/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs"
os.makedirs(main_out_dir, exist_ok=True)

In [15]:
model = "mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen"
threshold = "bdt1_0.333333_bdt2_0.366667_length_10"

In [7]:
DEFAULT_OVERRIDE_CONFIGS = [
    "override/livetime/hese_livetime_13yr_asr.cfg",
]
DEFAULT_OVERRIDE_COMPONENTS = [
    "override/components/astro_SPL_no_inel_no_flavor.yaml",
    "override/muon/muontemplate_hese_11features_plus_rloglmilli_econf_evtgen_bdt1_0.333333_bdt2_0.366667_paper.yaml",
]

DEFAULT_NO_SYSTEMATICS = ["override/systematics/NoSystematics_hese.cfg"]
DEFAULT_SYSTEMATICS = [f"override/systematics/hese/Snowstorm_Gradients_hese_HESEBestfit_SPL/{model}/{threshold}_wpriors.cfg"]

CONFIGS = []

In [8]:

CONFIGS += [
    {
        "name": f"11features_plus_rloglmilli_econf_evtgen",
        "analysis_config": f"analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen.yaml",
        "main_config" : "main.cfg",
        "override_configs": [f"override/datasets_hese/data_v3/{model}/{threshold}.cfg",
                             f"override/binning/hese/10bdtprod_threshold_0.122.cfg"] + DEFAULT_OVERRIDE_CONFIGS,
        "override_components": DEFAULT_OVERRIDE_COMPONENTS,
        "override_parameters": None,
    },
]

In [16]:
_NO_FLAVOR = ["override/components/astro_SPL_no_inel_no_flavor.yaml"]

_BDT_VARS = [
    ("bdt1_bdt2",               "bdt1_bdt2_10bins.cfg",                                                            DEFAULT_OVERRIDE_COMPONENTS),
    ("energy_length_analysis",  "energy_length_analysis.cfg",                                                        _NO_FLAVOR),
    ("len_easym",               "Taupede_Distance_Taupede_Asymmetry_10bins.cfg",                                     _NO_FLAVOR),
    ("e1_e2",                   "Taupede1_Particles_energy_Taupede2_Particles_energy_10bins.cfg",                    _NO_FLAVOR),
    ("e1_e2_zoom",              "Taupede1_Particles_energy_Taupede2_Particles_energy_10bins_zoom.cfg",               _NO_FLAVOR),
    ("mono_energy_zenith",      "MonopodFit_iMIGRAD_PPB0_energy_cscdSBU_MonopodFit4_noDC_zenith_10bins.cfg",       _NO_FLAVOR),
    ("mono_delay_qmax",         "MonopodFit_iMIGRAD_PPB0_Delay_ice_CVStatistics_q_max_doms_10bins.cfg",            _NO_FLAVOR),
    ("mono_delay_qmax_zoom",    "MonopodFit_iMIGRAD_PPB0_Delay_ice_CVStatistics_q_max_doms_10bins_zoom.cfg",       _NO_FLAVOR),
    ("vtxdist_qtot",            "cscdSBU_VertexRecoDist_CscdLLh_cscdSBU_Qtot_HLC_log_10bins.cfg",                 _NO_FLAVOR),
    ("vtxdist_qtot_zoom",       "cscdSBU_VertexRecoDist_CscdLLh_cscdSBU_Qtot_HLC_log_10bins_zoom.cfg",            _NO_FLAVOR),
    ("taumono_econf",           "TauMonoDiff_rlogl_econfinement_10bins.cfg",                                        _NO_FLAVOR),
    ("taumono_econf_zoom",      "TauMonoDiff_rlogl_econfinement_10bins_zoom.cfg",                                   _NO_FLAVOR),
    ("tauspe_taumilli",         "TauSPEMilliDiff_rlogl_TauMonoMilliDiff_rlogl_10bins.cfg",                        _NO_FLAVOR),
    ("tauspe_taumilli_zoom",    "TauSPEMilliDiff_rlogl_TauMonoMilliDiff_rlogl_10bins_zoom.cfg",                   _NO_FLAVOR),
    ("evtgen_recoeratio",       "EventGeneratorDC_Thijs_length_RecoERatio_EventGeneratorDC_Max_10bins.cfg",        _NO_FLAVOR),
    ("evtgen_recoeratio_zoom",  "EventGeneratorDC_Thijs_length_RecoERatio_EventGeneratorDC_Max_10bins_zoom.cfg",   _NO_FLAVOR),
]

In [17]:
_bdt_configs = []
for var, binning, components in _BDT_VARS:
    _comp = DEFAULT_OVERRIDE_COMPONENTS if components is not _NO_FLAVOR else _NO_FLAVOR
    _bdt_configs.append({
        "name": f"11features_plus_rloglmilli_econf_evtgen_{var}",
        "analysis_config": f"analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined.yaml",
        "main_config": "main.cfg",
        "override_configs": [
            "override/livetime/hese_livetime_13yr_asr.cfg",
            f"override/binning/hese/combined/{binning}",
            "override/binning/hese/combined/cut_energy.cfg",
            f"override/datasets_hese/data_v3/{model}/{threshold}.cfg",
        ],
        "syst_cfg":   f"override/systematics/hese_combined/debug_bdt/FinalTopology_double_energy_length_binning/hese_combined_HESEBestfit_SPL_no_flavor_{var}.cfg",
        "nosyst_cfg": "override/systematics/NoSystematics_hese_combined.cfg",
        "override_components": _comp,
        "override_parameters": None,
    })
CONFIGS += _bdt_configs

In [18]:
unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
import sys
sys.path.append(unblind_script_path)
from histogram_maker import *
from do_scan_analysis import *

In [19]:
for cfg in CONFIGS:
    name = cfg["name"]
    output_dir = os.path.join(main_out_dir, name)
    os.system(f"mkdir -p {output_dir}")
    outfile = os.path.join(output_dir,"Precalculated_Graph.pickle")

    print(10*"-")
    print(name)
    print(output_dir)
    print(outfile)

    # Each config may supply its own systematics config; fall back to the default.
    syst_cfgs   = [cfg["syst_cfg"]]   if "syst_cfg"   in cfg else DEFAULT_SYSTEMATICS
    nosyst_cfgs = [cfg["nosyst_cfg"]] if "nosyst_cfg" in cfg else DEFAULT_NO_SYSTEMATICS

    NNMFit_config_options = {
        "init_method": "from_configs",
        "main_config": os.path.join(configs_dir, cfg["main_config"]),
        "analysis_config": os.path.join(configs_dir, cfg["analysis_config"]),
        "config_dir": configs_dir,
        "override_configs": cfg["override_configs"] + syst_cfgs,
        "override_components": cfg["override_components"],
        "override_parameters": cfg["override_parameters"],
    }

    cmd = [
        python, script,
        "--main_config", os.path.join(configs_dir, cfg["main_config"]),
        "--analysis_config", os.path.join(configs_dir, cfg["analysis_config"]),
        "--config_dir", configs_dir,
        "--override_configs", *cfg["override_configs"] + syst_cfgs,
        "--override_components", *cfg["override_components"],
        "-o", outfile,
    ]
    if cfg["override_parameters"]:
        cmd += ["--override_parameters", *cfg["override_parameters"]]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    # Total histogram WITH systematics
    make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options, out_file=f"{output_dir}/MC_Histogram.pickle")
    generate_data_hist(scan_path=output_dir)

    # Total histogram WITHOUT systematics — used to compute per-bin gradient at plot time
    NNMFit_config_options_nosyst = NNMFit_config_options.copy()
    NNMFit_config_options_nosyst["override_configs"] = cfg["override_configs"] + nosyst_cfgs
    make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options_nosyst, out_file=f"{output_dir}/MC_Histogram_nosyst.pickle")

    # Per-component histograms WITHOUT systematics.
    # The per-bin gradient (total_syst - total_nosyst) is distributed among non-Muon
    # components at plot time weighted by component rate per bin (see plot_functions.py).
    for component in ["Astro", "Conv", "Muon"]:
        analysis_config_comp = os.path.join(configs_dir, cfg["analysis_config"].replace(".yaml", f"_{component}.yaml"))
        NNMFit_config_options_nosyst["analysis_config"] = analysis_config_comp
        print(10*"-", "Component (no syst)", component)
        print(analysis_config_comp)
        make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options_nosyst, out_file=f"{output_dir}/MC_Histogram_{component}_nosyst.pickle")

----------
11features_plus_rloglmilli_econf_evtgen
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen/Precalculated_Graph.pickle

Called with:
main_config             : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/main.cfg
analysis_config         : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen.yaml
config_dir              : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/
override_configs        : ['override/datasets_hese/data_v3/mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/bdt1_0.333333_bdt2_0.366667_length_10.cfg', 'override/binning/he

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-29 11:31:44; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-29 11:31:44; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:31:44; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-29 11:31:44; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:31:44; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-29 11:31:44; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2026-06-29 11:31:44; nnm_fitter.py:92)
DEBUG:NNMFit.core.nnm_fitter: Using ve

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_bdt1_bdt2/Data_Histogram.pickle
Full execution took 0.46917247772216797 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_bdt1_bdt2/Data_Histogram.pickle
---------- Component (no syst) Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Astro.yaml
---------- Component (no syst) Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Conv.yaml
---------- Component (no syst) Muon
/data/user/tvaneede/GlobalFit/reco_processi

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-29 11:32:01; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['reco_energy', 'reco_length'] (2026-06-29 11:32:01; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:32:01; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['reco_energy', 'reco_length'] (2026-06-29 11:32:01; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:32:01; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-29 11:32:01; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2026-06-29 11:32:01; nnm_fitter.py:92)
DEBUG:NNMFit.core.nnm_fitter: Using ve

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_energy_length_analysis/Data_Histogram.pickle
Full execution took 0.4387941360473633 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_energy_length_analysis/Data_Histogram.pickle
---------- Component (no syst) Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Astro.yaml
---------- Component (no syst) Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Conv.yaml
---------- Component (no syst) Muon
/data/user/tvaneed

...

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_tauspe_taumilli_zoom/Data_Histogram.pickle
Full execution took 0.4535789489746094 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_tauspe_taumilli_zoom/Data_Histogram.pickle
---------- Component (no syst) Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Astro.yaml
---------- Component (no syst) Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Conv.yaml
---------- Component (no syst) Muon
/data/user/tvaneede/Gl

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-29 11:35:38; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['EventGeneratorDC_Thijs_length', 'RecoERatio_EventGeneratorDC_Max'] (2026-06-29 11:35:38; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:35:38; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['EventGeneratorDC_Thijs_length', 'RecoERatio_EventGeneratorDC_Max'] (2026-06-29 11:35:38; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:35:38; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-29 11:35:38; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_evtgen_recoeratio/Data_Histogram.pickle
Full execution took 0.4630458354949951 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_evtgen_recoeratio/Data_Histogram.pickle
---------- Component (no syst) Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Astro.yaml
---------- Component (no syst) Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Conv.yaml
---------- Component (no syst) Muon
/data/user/tvaneede/GlobalFi

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-29 11:35:55; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['EventGeneratorDC_Thijs_length', 'RecoERatio_EventGeneratorDC_Max'] (2026-06-29 11:35:55; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:35:55; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['EventGeneratorDC_Thijs_length', 'RecoERatio_EventGeneratorDC_Max'] (2026-06-29 11:35:55; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-29 11:35:55; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-29 11:35:55; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_evtgen_recoeratio_zoom/Data_Histogram.pickle
Full execution took 0.458111047744751 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux/graphs/11features_plus_rloglmilli_econf_evtgen_evtgen_recoeratio_zoom/Data_Histogram.pickle
---------- Component (no syst) Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Astro.yaml
---------- Component (no syst) Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/11features_plus_rloglmilli_econf_evtgen/11features_plus_rloglmilli_econf_evtgen_combined_Conv.yaml
---------- Component (no syst) Muon
/data/user/tvaneede